# CS229 L04 — Exponential Family, GLMs & Softmax Regression

**Stanford CS229 · Spring 2026 · Instructor: Chris**  
[▶ Lecture Video](https://www.youtube.com/watch?v=8gVi4Rk21Eg) · [Official Notes](https://cs229.stanford.edu/notes/cs229-notes1.pdf) · [Course Website](https://cs229.stanford.edu/)

---

> 📌 *Lecture:* — instructor's exact words from the transcript  
> 🎯 **Interview:** — Q&A blocks for interview articulation

---

## 1. Why the Exponential Family?

> 📌 *Lecture:* "This is an idea that will allow us to generalize a lot of the linear models and things that we've seen to more complicated distributions. If you have a data type that's out there in the world — regression, classification, count data, all kinds of stuff — this gives you a reliable way of going from a distribution over that information to a learned model. That procedure you go through is going to be identical. All the things that you saw for linear least squares and logistic regression basically carry off to this entire large class of models."

**The MLE recipe from L03:** Pick a distribution → write likelihood → maximize log-likelihood → run SGD.

**The problem:** We derived least squares and cross-entropy separately. Do we need a new derivation for every new problem?

**The answer:** No. The exponential family unifies them all. Once you recognize your distribution belongs to this family, inference and learning are automatic.

> 📌 *Lecture:* "Once you think in terms of errors, then the rest of the model — at least for generalized linear models — kind of falls out. You didn't have to have one lecture on least squares, logistic regression, multinomial, Dirichlet processes, Poisson counts — they're all just one family once you get into them."

## 2. The Exponential Family — Canonical Form

A distribution belongs to the exponential family if it can be written as:

$$\boxed{p(y; \eta) = b(y) \exp\left(\eta^T T(y) - a(\eta)\right)}$$

| Symbol | Name | Meaning |
|---|---|---|
| $\eta$ | Natural parameter | The "free parameter" we learn — analogous to θ |
| $T(y)$ | Sufficient statistic | What we measure about y (usually just y itself) |
| $a(\eta)$ | Log-partition function | Normalization: ensures probabilities sum to 1 |
| $b(y)$ | Base measure | Depends only on y, not on η |

> 📌 *Lecture:* "This log-partition function — it looks like just a weird normalization, but as we'll see, almost all of the action is going to go on in here. Because the probability distribution adds up to one, you're going to have some term that sums or integrates over all the possible y's — that's captured here. That's why you have this minus sign. You should think about it: it's got to be a sum or integral over all the possible ways y could be realized."

**The key property of a(η) — mean and variance for free:**

$$\frac{\partial}{\partial \eta} a(\eta) = \mathbb{E}[T(y)]$$

$$\frac{\partial^2}{\partial \eta^2} a(\eta) = \text{Var}(T(y))$$

> 📌 *Lecture:* "When you differentiate the log, this term goes down underneath — that's the differential of the log. Inside this term you get the original term times T(y). That's exactly the expected value of T(y). This is what I mean when I say the action is embedded in this log partition function. It has all the outcomes, and if I differentiate it I get exactly the expectation."

**Why does this matter?**
- Mean is free: $\mathbb{E}[y] = a'(\eta)$
- Variance is free: $\text{Var}(y) = a''(\eta)$
- Convexity is guaranteed: $a''(\eta) = \text{Var}(y) \geq 0$ — the log-likelihood is always convex, so gradient descent always finds the global minimum

> 🎯 **Interview:** *What is the exponential family and why does it matter for ML?*  
> The exponential family is a class of distributions that can be written in the form $p(y;\eta) = b(y)\exp(\eta^T T(y) - a(\eta))$. It includes the Gaussian, Bernoulli, Multinomial, Poisson, Gamma, and many others. It matters for three reasons: (1) Once you identify your distribution as a member, inference (computing expectations) and learning (MLE) both follow the same mechanical recipe. (2) The log-partition function $a(\eta)$ encodes the mean and variance as its first and second derivatives — so properties of any member distribution are derivable automatically. (3) The log-likelihood is always convex for exponential family distributions, guaranteeing a unique global minimum for gradient descent.

## 3. Examples: Bernoulli and Gaussian as Exponential Family

### 3.1 Bernoulli → Logistic Regression

Bernoulli distribution: $p(y; \phi) = \phi^y(1-\phi)^{1-y}$, where $y \in \{0,1\}$

**Rewriting in exponential family form:**

Step 1: Take logs to expose the linear structure:
$$p(y;\phi) = \exp\left(y \log\phi + (1-y)\log(1-\phi)\right)$$

Step 2: Group terms by what they depend on:
$$= \exp\left(y \underbrace{\log\frac{\phi}{1-\phi}}_{\eta} + \underbrace{\log(1-\phi)}_{-a(\eta)}\right)$$

**Identifying the components:**
- $T(y) = y$ (sufficient statistic = y itself)
- $\eta = \log\frac{\phi}{1-\phi}$ (natural parameter = **log-odds**)
- $b(y) = 1$
- $a(\eta) = -\log(1-\phi) = \log(1 + e^\eta)$

**Inverting to get φ from η:**
$$\phi = \frac{1}{1 + e^{-\eta}} = \sigma(\eta) \quad \leftarrow \text{the sigmoid appears naturally!}$$

> 📌 *Lecture:* "That's a pretty interesting form if you look at it. It's got this $1 + e^{-\text{something}}$ form. We saw something like that in logistic regression. That'll come back a little bit later."

$$\boxed{\text{Bernoulli} \in \text{Exponential Family} \implies \text{Logistic Regression}}$$

### 3.2 Gaussian → Linear Regression

Gaussian: $p(y;\mu) = \frac{1}{\sqrt{2\pi}\sigma}\exp\left(-\frac{(y-\mu)^2}{2\sigma^2}\right)$ (treating σ as fixed)

Expanding:
$$= \frac{1}{\sqrt{2\pi}\sigma}\exp\left(-\frac{y^2}{2\sigma^2}\right) \cdot \exp\left(\frac{\mu}{\sigma^2}y - \frac{\mu^2}{2\sigma^2}\right)$$

**Identifying the components:**
- $\eta = \mu/\sigma^2$ (natural parameter)
- $T(y) = y$
- $b(y) = \frac{1}{\sqrt{2\pi}\sigma}\exp(-y^2/2\sigma^2)$
- $a(\eta) = \mu^2/2\sigma^2 = \eta^2\sigma^2/2$

**Check:** $a'(\eta) = \eta\sigma^2 = \mu = \mathbb{E}[y]$ ✓

$$\boxed{\text{Gaussian} \in \text{Exponential Family} \implies \text{Linear Regression}}$$

> 📌 *Lecture:* "The two distributions you've seen so far in the course — the error distributions — both fit into this form. The takeaway: once you pick a distribution and set η = θᵀx, the exponential family gives you the rest. You already know all the pieces of inference, you know the pieces of learning, you can generalize to whatever distribution or type of data you have."

### 3.3 The Full Exponential Family Zoo

| Distribution | Data type | Natural use case |
|---|---|---|
| Bernoulli | Binary (0/1) | Binary classification → logistic regression |
| Gaussian | Real-valued | Regression → least squares |
| Multinomial | Categorical (K classes) | Multiclass classification → softmax |
| Poisson | Count (0,1,2,...) | Event counts (clicks, arrivals) |
| Gamma / Exponential | Positive real | Durations, waiting times |
| Dirichlet | Distribution over distributions | Topic modeling (LDA) |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import bernoulli, norm, poisson

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Bernoulli
phi = 0.7
axes[0].bar([0, 1], [1-phi, phi], color=['coral', 'steelblue'])
axes[0].set_title(f'Bernoulli(φ={phi})\nη = log(φ/(1-φ)) = {np.log(phi/(1-phi)):.2f}')
axes[0].set_xlabel('y');  axes[0].set_ylabel('p(y)')
axes[0].set_xticks([0, 1])

# Gaussian
x = np.linspace(-4, 4, 300)
for mu, color in [(-1, 'coral'), (0, 'steelblue'), (2, 'green')]:
    axes[1].plot(x, norm.pdf(x, mu, 1), label=f'μ={mu}, η=μ/σ²={mu}', color=color)
axes[1].set_title('Gaussian N(μ, 1)\nη = μ/σ²')
axes[1].set_xlabel('y'); axes[1].legend(fontsize=8)

# Poisson (new member — not in L02-L03)
k = np.arange(0, 12)
for lam, color in [(1, 'coral'), (3, 'steelblue'), (6, 'green')]:
    axes[2].plot(k, poisson.pmf(k, lam), 'o-', label=f'λ={lam}', color=color)
axes[2].set_title('Poisson(λ) — count data\nη = log(λ)')
axes[2].set_xlabel('y (count)'); axes[2].legend(fontsize=8)

plt.suptitle('Exponential Family Members', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()
print("All three are exponential family — same learning recipe for all.")

## 4. Generalized Linear Models (GLMs)

GLMs tie the exponential family to actual supervised learning with features X.

### 4.1 The Three GLM Assumptions

> 📌 *Lecture:* "We're going to pick a distribution based on Y's type. Binary → Bernoulli. Multiple classes → Multinomial. And then our model is going to be linear — we're going to set the natural parameter η = θᵀx. Theta is a parameter that we learn. Theta together with an X data point creates the natural parameter η of the noise."

1. **Label distribution:** $y | x; \theta \sim \text{ExponentialFamily}(\eta)$ — choose based on y's type
2. **Linear predictor:** $\eta = \theta^T x$ — the natural parameter is linear in x
3. **Prediction:** $h_\theta(x) = \mathbb{E}[y | x; \theta]$ — predict the expected value

### 4.2 The GLM Recipe

```
Step 1: Look at y's type → choose distribution (Bernoulli, Gaussian, Poisson, ...)
Step 2: Write that distribution in exponential family form → identify a(η)
Step 3: Set η = θᵀx
Step 4: Prediction: h_θ(x) = E[y|x;θ] = a'(θᵀx)   [derivative of log-partition]
Step 5: Training: maximize log-likelihood with SGD
```

### 4.3 The Universal SGD Update

> 📌 *Lecture:* "The algorithm always has this really fun form which you can verify: I look at the misprediction error, I multiply it by the data, and I move in that direction for theta. All the things that we derived one after the other as consequences of logistic regression or least squares are actually just instances of a more general algorithm. And this algorithm works no matter how you pick the errors."

For **every** GLM, the SGD update is:
$$\theta_j := \theta_j + \alpha \left(y^{(i)} - h_\theta(x^{(i)})\right) x_j^{(i)}$$

**Error × feature — always.** The specific form of $h_\theta$ changes (sigmoid for logistic, linear for regression, softmax for multiclass), but the update rule is identical.

| Model | Distribution | $h_\theta(x)$ | Loss |
|---|---|---|---|
| Linear Regression | Gaussian | $\theta^T x$ | Squared loss |
| Logistic Regression | Bernoulli | $\sigma(\theta^T x)$ | Cross-entropy |
| Softmax Regression | Multinomial | $\text{softmax}(\Theta x)$ | Cross-entropy |
| Poisson Regression | Poisson | $e^{\theta^T x}$ | Poisson log-loss |

### 4.4 Parameter Naming (The Confusing Part)

> 📌 *Lecture:* "This is the part that I highlight because it drives everyone crazy. There are model parameters — those are the thetas — that's what you learn. Theta together with X turns into a natural parameter in our formulation. That turns by a link function into the canonical parameters. These are the only ones you learn. The rest of this is modeling the noise."

$$\underbrace{\theta}_{\text{model weights}} \xrightarrow{\theta^T x} \underbrace{\eta}_{\text{natural param}} \xrightarrow{g^{-1}(\cdot)} \underbrace{\phi, \mu, \lambda, \ldots}_{\text{canonical params}}$$

Only **θ** is learned. η and the canonical params are derived from θ and x.

> 🎯 **Interview:** *What is a Generalized Linear Model? How does it unify logistic and linear regression?*  
> A GLM has three components: (1) a distribution from the exponential family for y|x, (2) a linear predictor η = θᵀx, and (3) a link function connecting η to the distribution's canonical parameter. Linear regression is the GLM with Gaussian noise — η = μ = θᵀx, link is identity. Logistic regression is the GLM with Bernoulli labels — η = log(φ/(1-φ)), link is sigmoid. What's unified: the SGD update `θ := θ + α(y - h_θ(x))x` has the same form for every GLM. You're always doing error × feature. Only the prediction function h_θ changes.

## 5. Softmax Regression — Multiclass Classification

> 📌 *Lecture:* "This is the thing that you probably use the most. Every time one of those AI models runs, the last step that it does is it picks among all the tokens in its vocabulary which one is the most likely to generate next. And it does that with a softmax. Also underneath the covers, this multiclass classification shows up in attention — the workhorse of the transformer architecture."

### 5.1 Setup

**Goal:** classify into $K$ classes. $y \in \{1, 2, ..., K\}$.

**One-hot encoding:** represent each class as a unit vector:
$$y = \text{cat} \to [1,0,0,0], \quad y = \text{dog} \to [0,1,0,0], \quad \ldots$$

**Prediction:** a probability distribution over all K classes — a vector summing to 1:
$$h_\Theta(x) = [P(y=1|x), P(y=2|x), \ldots, P(y=K|x)]$$

**Parameters:** $\Theta \in \mathbb{R}^{K \times d}$ — one weight vector $\theta_k$ per class.

### 5.2 The Softmax Function

$$P(y=k | x; \Theta) = \frac{\exp(\theta_k^T x)}{\sum_{j=1}^{K} \exp(\theta_j^T x)}$$

> 📌 *Lecture:* "For every class, for every theta j, you dot it into X and think about it like a bake-off. It bakes off through this exponential model. The exponential is nice because it has smooth properties. If things are very clearly separated from the region you care about, they're going to get crushed by the exponential — that's one of the things you like: you get that sigmoid behavior."

**Properties of softmax:**
- $P(y=k|x) \in (0,1)$ for all k — always valid probabilities
- $\sum_k P(y=k|x) = 1$ — sums to 1
- Smooth and differentiable everywhere
- Invariant to adding a constant to all logits: $\text{softmax}(z+c) = \text{softmax}(z)$
- K=2 case reduces exactly to logistic regression

**The geometric picture:**

> 📌 *Lecture:* "I can separate all these blue cats by drawing a line — a hyperplane because in higher dimensions it's not just a line. That hyperplane says if you're on this side, you're a cat. I have another one for dogs: theta 2 dot x = 0. I draw them for each class. In cleanly separated data, I can draw nice hyperplanes. What happens in regions that are contentious? There I have to bake them off against each other — and that's what softmax does."

### 5.3 K=2 Reduces to Logistic Regression

With K=2:
$$P(y=1|x) = \frac{e^{\theta_1^T x}}{e^{\theta_1^T x} + e^{\theta_2^T x}} = \frac{1}{1 + e^{-(\theta_1-\theta_2)^T x}} = \sigma((\theta_1-\theta_2)^T x)$$

Setting $\theta = \theta_1 - \theta_2$, this is exactly logistic regression. Softmax is the correct generalization.

In [ ]:
def softmax(z):
    # Numerically stable: subtract max before exp
    z = z - z.max(axis=-1, keepdims=True)
    exp_z = np.exp(z)
    return exp_z / exp_z.sum(axis=-1, keepdims=True)

# Demonstrate softmax bake-off
scores = np.array([
    [3.0, 1.0, 0.2, -1.0],   # clearly cat
    [0.5, 2.5, 0.1, 0.3],    # clearly dog
    [1.0, 1.0, 3.0, 0.5],    # clearly car
    [0.8, 1.2, 0.6, 0.9],    # ambiguous
])
probs = softmax(scores)
classes = ['cat', 'dog', 'car', 'bus']

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Raw scores vs probabilities
x_pos = np.arange(len(classes))
for i, (label, score_row, prob_row) in enumerate(zip(
    ['Clearly cat', 'Clearly dog', 'Clearly car', 'Ambiguous'],
    scores, probs
)):
    axes[0].bar(x_pos + i*0.2, score_row, width=0.18, label=label, alpha=0.8)
    axes[1].bar(x_pos + i*0.2, prob_row, width=0.18, label=label, alpha=0.8)

for ax, title, ylabel in [
    (axes[0], 'Raw Scores θⱼᵀx (logits)', 'Score'),
    (axes[1], 'After Softmax — Probabilities', 'P(y=k|x)')
]:
    ax.set_title(title); ax.set_ylabel(ylabel)
    ax.set_xticks(x_pos + 0.3); ax.set_xticklabels(classes)
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

print("Scores → Softmax probabilities:")
for label, prob_row in zip(['Clearly cat', 'Clearly dog', 'Clearly car', 'Ambiguous'], probs):
    pred = classes[np.argmax(prob_row)]
    conf = prob_row.max()
    print(f"  {label:15s}: {dict(zip(classes, prob_row.round(3)))} → {pred} ({conf:.1%})")

## 6. Cross-Entropy Loss for Softmax

### 6.1 Derivation from MLE

Multinomial likelihood (one training example, true class k*):
$$P(y | x; \Theta) = \prod_{k=1}^{K} P(y=k|x)^{\mathbf{1}[y=k]}$$

Log-likelihood (using one-hot vector $p$ where $p_{k^*}=1$, rest 0):
$$\ell(\Theta) = \sum_{k=1}^{K} p_k \log P(y=k|x)$$

Only the true class term survives (others have $p_k = 0$):
$$\ell(\Theta) = \log P(y=k^*|x) = \log \text{softmax}(\theta_{k^*}^T x)$$

**Cross-entropy loss** (negate, sum over training set):
$$\boxed{J(\Theta) = -\frac{1}{n}\sum_{i=1}^{n} \sum_{k=1}^{K} \mathbf{1}[y^{(i)}=k] \log P(y=k|x^{(i)}; \Theta)}$$

> 📌 *Lecture:* "Cross entropy — we'll derive exactly from this. It's just taking logs across exactly this form. There's a privileged p-hat here that comes out exactly from the label — that's a one-hot vector. Right now only one of these terms survives. What's saying is I imagine that I'm basically saying whichever one gives me the best score, I'm going to try and optimize."

### 6.2 Label Smoothing

> 📌 *Lecture:* "Label smoothing says: instead of having the true distribution be exactly one here, take that mass and spread it out. Say there's $1-2\epsilon$ here and then $\epsilon$ on the neighbors. Intuitively why might you want to do that? This performs a type of regularization. It prevents you from really overfitting and driving the model to exactly predict one class. Adding a little label noise makes the model a little bit more honest."

**Hard labels (standard):** $p = [0, 0, 1, 0]$ (one-hot)

**Soft labels (label smoothing, ε=0.1):**
$$p_k = \begin{cases} 1 - \epsilon(K-1)/K & \text{if } k = k^* \\ \epsilon/K & \text{otherwise} \end{cases}$$

Example with K=4, ε=0.1: $p \approx [0.025, 0.025, 0.925, 0.025]$

> 🎯 **Interview:** *Why is softmax + cross-entropy used for multiclass classification?*  
> Softmax comes from MLE under a Multinomial distribution — the same way sigmoid came from Bernoulli. The cross-entropy loss is the negative log-likelihood of the true class. Together they have three nice properties: (1) Softmax always outputs valid probabilities (positive, sum to 1). (2) The gradient of cross-entropy + softmax has a clean form: $\hat{p} - p$ — predicted distribution minus true distribution. (3) The exponential in softmax means distant wrong answers get crushed, but nearby ones still influence the gradient, giving smooth learning signal. This is why softmax + cross-entropy is the last layer of every LLM and image classifier.

In [ ]:
def cross_entropy(probs, labels):
    eps = 1e-9
    return -np.mean(np.sum(labels * np.log(probs + eps), axis=1))

def softmax_regression(X, Y, lr=0.5, n_iter=500):
    n, d = X.shape
    K = Y.shape[1]
    Theta = np.zeros((K, d))
    losses = []
    for _ in range(n_iter):
        logits = X @ Theta.T           # (n, K)
        probs = softmax(logits)        # (n, K)
        error = probs - Y              # (n, K)  predicted - true
        grad = error.T @ X / n        # (K, d)
        Theta -= lr * grad
        losses.append(cross_entropy(probs, Y))
    return Theta, losses

# Generate 4-class data
np.random.seed(42)
K, n_per_class = 4, 100
centers = np.array([[2,2], [-2,2], [-2,-2], [2,-2]])
X_all = np.vstack([np.random.randn(n_per_class, 2) * 0.7 + c for c in centers])
y_all = np.repeat(np.arange(K), n_per_class)

# One-hot encode
Y_all = np.eye(K)[y_all]

# Add bias
X_bias = np.hstack([np.ones((len(X_all), 1)), X_all])

Theta, losses = softmax_regression(X_bias, Y_all, lr=0.5, n_iter=300)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Decision regions
xx, yy = np.meshgrid(np.linspace(-5, 5, 200), np.linspace(-5, 5, 200))
XY = np.c_[np.ones(xx.ravel().shape), xx.ravel(), yy.ravel()]
Z = np.argmax(softmax(XY @ Theta.T), axis=1).reshape(xx.shape)
axes[0].contourf(xx, yy, Z, levels=[-0.5, 0.5, 1.5, 2.5, 3.5],
                 colors=['#AED6F1', '#A9DFBF', '#F9E79F', '#F1948A'], alpha=0.4)
colors = ['steelblue', 'green', 'orange', 'red']
labels = ['cat', 'dog', 'car', 'bus']
for k in range(K):
    mask = y_all == k
    axes[0].scatter(X_all[mask,0], X_all[mask,1], c=colors[k], label=labels[k], s=20, alpha=0.7)
axes[0].set_title('Softmax Regression — 4-Class Decision Regions')
axes[0].legend()

# Loss curve
axes[1].plot(losses)
axes[1].set_xlabel('Iteration'); axes[1].set_ylabel('Cross-Entropy Loss')
axes[1].set_title('Training Loss')

plt.tight_layout()
plt.show()

preds = np.argmax(softmax(X_bias @ Theta.T), axis=1)
print(f"Accuracy: {(preds == y_all).mean()*100:.1f}%")
print(f"Final loss: {losses[-1]:.4f}")
print(f"Gradient form: (predicted_probs - true_one_hot) × x  [error × feature, always]")

## 7. The Big Picture — What L04 Unlocks

> 📌 *Lecture:* "We saw that cross entropy and softmax, which I showed you are techniques that are foundational — this is stuff that you have probably used in a product within the last hour. I mentioned label smoothing because I also wanted to highlight this point: your data and model are always a little bit wrong. Often in machine learning, it's looking for the more robust thing to do. And then I emphasize this: this exact setup — softmax with cross entropy — is underneath all these amazing AIs. Every token one of those models generates came from multiclass classification."

**What the exponential family gives you for free:**

| You provide | You get automatically |
|---|---|
| Choice of distribution (Bernoulli, Gaussian, ...) | The loss function (cross-entropy, squared loss, ...) |
| Set η = θᵀx | The prediction function h_θ(x) |
| IID assumption | SGD update: (y - h_θ(x)) · x |
| Exponential family form | Convex log-likelihood (guaranteed convergence) |

**Connection to modern AI:**

```
LLM last layer:  logits = Wh_t     (linear, W ∈ R^{vocab × d})
                 probs = softmax(logits)
                 loss  = cross_entropy(probs, true_token)
```

This is exactly GLM with Multinomial distribution. CS229 L04 → GPT-4's output layer.

> 🎯 **Interview:** *What is the connection between logistic regression, softmax, and modern LLMs?*  
> All three are instances of GLMs with exponential family distributions. Logistic regression is a 2-class GLM with Bernoulli labels — predict P(y=1|x) with sigmoid. Softmax regression generalizes to K classes with Multinomial labels. LLMs use softmax regression as their output layer: at each token position, the model produces logits (one per vocabulary token), applies softmax to get a probability distribution, and minimizes cross-entropy loss against the true next token. The math of L04 is literally the last layer of every large language model.

## External Resources

| Resource | What it covers | When to use |
|---|---|---|
| [CS229 Notes 1](https://cs229.stanford.edu/notes/cs229-notes1.pdf) | GLMs, exponential family derivations | Primary reference — sections 4–6 |
| [StatQuest — Softmax](https://www.youtube.com/watch?v=8ah-qul37VU) | Visual softmax explanation | Before coding softmax from scratch |
| [Understanding Deep Learning — Prince Ch. 5](https://udlbook.github.io/udlbook/) | Cross-entropy and loss functions | Modern framing of GLMs |
| [The Annotated Transformer](https://nlp.seas.harvard.edu/annotated-transformer/) | Softmax in attention mechanism | See how softmax extends to L14 |
| [Label Smoothing paper (Szegedy 2016)](https://arxiv.org/abs/1512.00567) | Rethinking Inception — introduced label smoothing | Original label smoothing paper |